In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df_mail=pd.read_csv("DataSet\spam.csv",encoding="latin")


In [18]:
df_mail.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [19]:
df_mail.drop(["Unnamed: 2","Unnamed: 3","Unnamed: 4"],axis=1,inplace=True)

In [20]:
df_mail

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [21]:
df_mail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [22]:
df_mail["v1"]=df_mail["v1"].map({"ham":1,"spam":0})

In [23]:
df_mail

,v1,v2
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,0,This is the 2nd time we have tried 2 contact u...
5568,1,Will Ì_ b going to esplanade fr home?
5569,1,"Pity, * was in mood for that. So...any other s..."
5570,1,The guy did some bitching but I acted like i'd...


In [24]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\razas\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [25]:
def clean_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]','',text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+','',text)
    text = re.sub('<.*?>+',b'',text)
    text = re.sub('[%s]' % re.escape(string.punctuation),'',text)
    text = re.sub('\w*\d\w*','',text)
    return text

In [26]:
df_mail["v2"]=df_mail["v2"].apply(clean_text)

In [27]:
df_mail

,v1,v2
0,1,go until jurong point crazy available only ...
1,1,ok lar joking wif u oni
2,0,free entry in a wkly comp to win fa cup final...
3,1,u dun say so early hor u c already then say
4,1,nah i don t think he goes to usf he lives aro...
...,...,...
5567,0,this is the time we have tried contact u u ...
5568,1,will ì b going to esplanade fr home
5569,1,pity was in mood for that so any other s...
5570,1,the guy did some bitching but i acted like i d...


In [28]:
stopwords=set(stopwords.words("english"))
df_mail["v2"]=df_mail["v2"].apply(lambda x: " ".join(word for word in x.split() if word not in stopwords))

In [29]:
df_mail

,v1,v2
0,1,go jurong point crazy available bugis n great ...
1,1,ok lar joking wif u oni
2,0,free entry wkly comp win fa cup final tkts may...
3,1,u dun say early hor u c already say
4,1,nah think goes usf lives around though
...,...,...
5567,0,time tried contact u u å pound prize claim eas...
5568,1,ì b going esplanade fr home
5569,1,pity mood suggestions
5570,1,guy bitching acted like interested buying some...


In [30]:
X=df_mail["v2"]
y=df_mail["v1"]

In [31]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [32]:
vectorize=TfidfVectorizer()

In [33]:
vectorize.fit(X_train)
X_train_vect=vectorize.transform(X_train)
X_test_vect=vectorize.transform(X_test)

In [34]:
def create_model(X_train_vect,y_train,X_test_vect,y_test):
  model={
      "SVC":SVC(),
      "RandomForestClassifier":RandomForestClassifier(),
      "MultinomialNB":MultinomialNB(),
      "LogisticRegression":LogisticRegression(),
      "XGBClassifier":XGBClassifier(),
      "DecisionTreeClassifier":DecisionTreeClassifier(),
      "GradientBoostingClassifier":GradientBoostingClassifier()
  }
  for name,m in model.items():
    m.fit(X_train_vect,y_train)
    y_pred=m.predict(X_test_vect)
    print(f"Accuracy score of {name} is {accuracy_score(y_test,y_pred)}")


In [35]:
create_model(X_train_vect,y_train,X_test_vect,y_test)

Accuracy score of SVC is 0.9748878923766816
Accuracy score of RandomForestClassifier is 0.9757847533632287
Accuracy score of MultinomialNB is 0.9659192825112107
Accuracy score of LogisticRegression is 0.9533632286995516
Accuracy score of XGBClassifier is 0.979372197309417
Accuracy score of DecisionTreeClassifier is 0.957847533632287
Accuracy score of GradientBoostingClassifier is 0.9659192825112107


In [ ]:
import joblib
joblib.dump(XGBClassifier,"spam_email_detector_model.pkl")

['spam_detection.pkl']